# 03 — LightGBM

Standalone notebook. Run `00_Preprocessing.ipynb` first.

In [ ]:
import os, time
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier

from utils import PROCESSED_DIR, RESULTS_DIR, MODELS_DIR, PLOTS_DIR, evaluate, save_results

SEED = 42
MODEL_NAME = "LightGBM"


## Load Preprocessed Data

In [ ]:
X_train_res = np.load(os.path.join(PROCESSED_DIR, "X_train_res.npy"))
y_train_res = np.load(os.path.join(PROCESSED_DIR, "y_train_res.npy"))
X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))
class_names = joblib.load(os.path.join(PROCESSED_DIR, "class_names.joblib"))
n_classes = joblib.load(os.path.join(PROCESSED_DIR, "n_classes.joblib"))

print(f"Train: {X_train_res.shape}, Test: {X_test.shape}, classes: {n_classes}")


## Train

In [ ]:
model = LGBMClassifier(
    n_estimators=300, max_depth=-1, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9,
    objective="multiclass", num_class=n_classes,
    n_jobs=-1, random_state=SEED, verbose=-1,
)

t0 = time.time()
model.fit(X_train_res, y_train_res)
train_time = time.time() - t0
print(f"Train time: {train_time:.2f}s")


## Predict & Evaluate

In [ ]:
t0 = time.time()
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
infer_time = time.time() - t0

row = evaluate(MODEL_NAME, y_test, y_pred, y_proba, train_time, infer_time, n_classes)
for k, v in row.items():
    if k not in ("Confusion Matrix",):
        print(f"{k}: {v}")


## Confusion Matrix Plot

In [ ]:
cm = np.array(row["Confusion Matrix"])
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"{MODEL_NAME} — Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "LightGBM_confusion_matrix.png"), dpi=150)
plt.show()


## Save Results & Model

In [ ]:
save_results(MODEL_NAME, row)
joblib.dump(model, os.path.join(MODELS_DIR, "LightGBM.joblib"))
print("Saved model and results for", MODEL_NAME)
